# Stress Prediction V31 — Absolute-Error ExtraTrees Blend

기존 squared-error ExtraTrees와 absolute-error ExtraTrees를 Near-Full 검증으로 비교·혼합합니다. Test는 최종 예측에만 사용합니다.


In [1]:
from __future__ import annotations

import argparse
import os
from itertools import combinations
from pathlib import Path

os.environ.setdefault("OMP_NUM_THREADS", "1")
os.environ.setdefault("OPENBLAS_NUM_THREADS", "1")
os.environ.setdefault("MKL_NUM_THREADS", "1")
os.environ.setdefault("LOKY_MAX_CPU_COUNT", "1")

import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import ExtraTreesRegressor
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.metrics import mean_absolute_error
from sklearn.model_selection import KFold


TARGET = "stress_score"
ID_COLUMN = "ID"
RANDOM_SEED = 42
N_ESTIMATORS = 1200
TREE_QUANTILE = 0.52
PAIR_QUANTILE = 0.52
PAIR_WEIGHT = 0.15

FEATURE_COPY_COUNTS = {
    "mean_working": 1,
    "bmi": 4,
    "cholesterol": 2,
    "height": 2,
    "glucose": 3,
    "weight": 2,
    "cholesterol_glucose_ratio": 2,
    "bone_density": 4,
}

PAIR_COLUMNS = [
    "mean_working",
    "bmi",
    "cholesterol",
    "height",
    "glucose",
    "weight",
    "cholesterol_glucose_ratio",
    "bone_density",
]


def add_features(frame: pd.DataFrame) -> pd.DataFrame:
    """다른 행이나 Test 전체 통계를 사용하지 않는 행 단위 파생변수."""
    result = frame.copy()
    height_m = result["height"] / 100.0
    sbp = result["systolic_blood_pressure"]
    dbp = result["diastolic_blood_pressure"]
    result["bmi"] = result["weight"] / height_m.pow(2)
    result["pulse_pressure"] = sbp - dbp
    result["mean_arterial_pressure"] = (sbp + 2.0 * dbp) / 3.0
    result["blood_pressure_ratio"] = sbp / (dbp + 1e-6)
    result["cholesterol_glucose_ratio"] = result["cholesterol"] / (
        result["glucose"] + 1e-6
    )
    result["missing_count"] = result.isna().sum(axis=1)
    return result


def add_feature_copies(frame: pd.DataFrame) -> pd.DataFrame:
    result = frame.copy()
    for feature, additional_copies in FEATURE_COPY_COUNTS.items():
        for copy_index in range(1, additional_copies + 1):
            result[f"{feature}__copy{copy_index}"] = result[feature]
    return result


def make_preprocessor(frame: pd.DataFrame) -> ColumnTransformer:
    categorical = frame.select_dtypes(
        include=["object", "category", "string"]
    ).columns.tolist()
    numerical = frame.select_dtypes(include=["number"]).columns.tolist()
    return ColumnTransformer(
        [
            (
                "categorical",
                Pipeline(
                    [
                        (
                            "imputer",
                            SimpleImputer(
                                strategy="constant", fill_value="missing"
                            ),
                        ),
                        (
                            "encoder",
                            OneHotEncoder(
                                handle_unknown="ignore", sparse_output=False
                            ),
                        ),
                    ]
                ),
                categorical,
            ),
            (
                "numerical",
                SimpleImputer(strategy="median", add_indicator=True),
                numerical,
            ),
        ]
    )


def tree_quantile_prediction(
    train_features: pd.DataFrame,
    test_features: pd.DataFrame,
    target: pd.Series,
) -> np.ndarray:
    weighted_train = add_feature_copies(train_features)
    weighted_test = add_feature_copies(test_features)
    pipeline = Pipeline(
        [
            ("preprocessor", make_preprocessor(weighted_train)),
            (
                "model",
                ExtraTreesRegressor(
                    n_estimators=N_ESTIMATORS,
                    min_samples_leaf=1,
                    max_features=1,
                    random_state=RANDOM_SEED,
                    n_jobs=1,
                ),
            ),
        ]
    )
    pipeline.fit(weighted_train, target)
    matrix = pipeline.named_steps["preprocessor"].transform(weighted_test)
    forest = pipeline.named_steps["model"]
    tree_predictions = np.column_stack(
        [tree.predict(matrix) for tree in forest.estimators_]
    )
    return np.clip(
        np.round(np.quantile(tree_predictions, TREE_QUANTILE, axis=1), 2),
        0.0,
        1.0,
    )


def empirical_rank_transform(
    train_values: np.ndarray,
    test_values: np.ndarray,
) -> tuple[np.ndarray, np.ndarray]:
    """Train 분포만으로 Train과 한 개씩 독립적인 Test 값을 변환."""
    median = np.nanmedian(train_values)
    train_values = np.nan_to_num(train_values, nan=median)
    test_values = np.nan_to_num(test_values, nan=median)
    sorted_train = np.sort(train_values)
    train_rank = np.searchsorted(
        sorted_train, train_values, side="right"
    ) / len(train_values)
    test_rank = np.searchsorted(
        sorted_train, test_values, side="right"
    ) / len(train_values)
    return train_rank.astype(np.float32), test_rank.astype(np.float32)


def pair_neighbor_prediction(
    train_features: pd.DataFrame,
    test_features: pd.DataFrame,
    target: pd.Series,
) -> np.ndarray:
    """각 2차원 피처쌍에서 가장 가까운 Train target을 분위수 집계."""
    train_rank = {}
    test_rank = {}
    for column in PAIR_COLUMNS:
        train_rank[column], test_rank[column] = empirical_rank_transform(
            train_features[column].to_numpy(dtype=float),
            test_features[column].to_numpy(dtype=float),
        )

    target_values = target.to_numpy(dtype=float)
    neighbor_targets = []
    for first, second in combinations(PAIR_COLUMNS, 2):
        distance = (
            np.abs(test_rank[first][:, None] - train_rank[first][None, :])
            + np.abs(test_rank[second][:, None] - train_rank[second][None, :])
        )
        nearest_index = np.argmin(distance, axis=1)
        neighbor_targets.append(target_values[nearest_index])
    return np.quantile(
        np.column_stack(neighbor_targets), PAIR_QUANTILE, axis=1
    )


def validate_inputs(
    train: pd.DataFrame,
    test: pd.DataFrame,
    submission: pd.DataFrame,
) -> None:
    if set(train.columns) - {TARGET} != set(test.columns):
        raise ValueError("Train과 Test의 입력 컬럼이 다릅니다.")
    if list(submission.columns) != [ID_COLUMN, TARGET]:
        raise ValueError("제출 양식 컬럼이 올바르지 않습니다.")
    if not submission[ID_COLUMN].equals(test[ID_COLUMN]):
        raise ValueError("제출 양식과 Test의 ID 순서가 다릅니다.")
    if train[TARGET].isna().any():
        raise ValueError("Train 타깃에 결측값이 있습니다.")

In [2]:
from sklearn.model_selection import ShuffleSplit

DATA_DIR = Path("/Users/bitsaem/Desktop/stress_project/open")
OUTPUT_DIR = Path("/Users/bitsaem/Desktop/stress_project")
train = pd.read_csv(DATA_DIR / "train.csv")
test = pd.read_csv(DATA_DIR / "test.csv")
submission = pd.read_csv(DATA_DIR / "sample_submission.csv")
validate_inputs(train, test, submission)

X = add_features(train.drop(columns=[TARGET, ID_COLUMN])).reset_index(drop=True)
X_test = add_features(test.drop(columns=[ID_COLUMN])).reset_index(drop=True)
y = train[TARGET].reset_index(drop=True)

N_SPLITS = 8
VALID_SIZE = 150
AUDIT_TREES = 600
FINAL_TREES = 1200
TREE_QUANTILES = (0.48, 0.50, 0.52, 0.54)
ABS_WEIGHTS = (0.25, 0.50, 0.75, 1.00)


def tree_matrix(train_features, valid_features, target, criterion, trees):
    weighted_train = add_feature_copies(train_features)
    weighted_valid = add_feature_copies(valid_features)
    preprocessor = make_preprocessor(weighted_train)
    train_matrix = preprocessor.fit_transform(weighted_train)
    valid_matrix = preprocessor.transform(weighted_valid)
    model = ExtraTreesRegressor(
        n_estimators=trees,
        criterion=criterion,
        min_samples_leaf=1,
        max_features=1,
        random_state=42,
        n_jobs=-1,
    )
    model.fit(train_matrix, target)
    return np.column_stack(
        [tree.predict(valid_matrix) for tree in model.estimators_]
    )


def final_blend(tree_prediction, pair_prediction):
    return np.clip(
        np.round(0.85 * tree_prediction + 0.15 * pair_prediction, 2),
        0.0, 1.0,
    )


splitter = ShuffleSplit(
    n_splits=N_SPLITS,
    test_size=VALID_SIZE,
    random_state=31082026,
)
rows = []

for split_id, (train_idx, valid_idx) in enumerate(splitter.split(X), 1):
    fold_train = X.iloc[train_idx]
    fold_valid = X.iloc[valid_idx]
    fold_y = y.iloc[train_idx]
    valid_y = y.iloc[valid_idx]

    pair_pred = pair_neighbor_prediction(fold_train, fold_valid, fold_y)
    squared_matrix = tree_matrix(
        fold_train, fold_valid, fold_y, "squared_error", AUDIT_TREES
    )
    absolute_matrix = tree_matrix(
        fold_train, fold_valid, fold_y, "absolute_error", AUDIT_TREES
    )
    baseline_tree = np.quantile(squared_matrix, 0.52, axis=1)
    baseline_pred = final_blend(baseline_tree, pair_pred)
    baseline_mae = mean_absolute_error(valid_y, baseline_pred)

    for quantile in TREE_QUANTILES:
        squared_pred = np.quantile(squared_matrix, quantile, axis=1)
        absolute_pred = np.quantile(absolute_matrix, quantile, axis=1)
        for abs_weight in ABS_WEIGHTS:
            tree_pred = (
                (1.0 - abs_weight) * squared_pred
                + abs_weight * absolute_pred
            )
            candidate_pred = final_blend(tree_pred, pair_pred)
            candidate_mae = mean_absolute_error(valid_y, candidate_pred)
            rows.append({
                "split": split_id,
                "quantile": quantile,
                "absolute_weight": abs_weight,
                "baseline_MAE": baseline_mae,
                "candidate_MAE": candidate_mae,
                "improvement": baseline_mae - candidate_mae,
            })
    print(f"V31 Near-Full split {split_id}/{N_SPLITS} complete")

score_frame = pd.DataFrame(rows)
summary = (
    score_frame.groupby(["quantile", "absolute_weight"])
    .agg(
        candidate_MAE=("candidate_MAE", "mean"),
        mean_improvement=("improvement", "mean"),
        worst_improvement=("improvement", "min"),
        wins=("improvement", lambda value: int((value > 0).sum())),
    )
    .reset_index()
    .sort_values(
        ["wins", "worst_improvement", "candidate_MAE"],
        ascending=[False, False, True],
    )
    .reset_index(drop=True)
)
display(summary)

eligible = summary[
    (summary.wins >= 6)
    & (summary.mean_improvement >= 0.0003)
    & (summary.worst_improvement >= -0.0010)
]

if eligible.empty:
    print("V31 rejected: absolute-error 후보가 안정성 기준을 통과하지 못했습니다.")
    print("제출 파일을 생성하지 않습니다.")
    v31_accepted = False
else:
    selected = eligible.iloc[0]
    selected_quantile = float(selected["quantile"])
    selected_weight = float(selected["absolute_weight"])
    v31_accepted = True
    print("V31 accepted")
    print(selected)

    pair_test = pair_neighbor_prediction(X, X_test, y)
    squared_test = tree_matrix(X, X_test, y, "squared_error", FINAL_TREES)
    absolute_test = tree_matrix(X, X_test, y, "absolute_error", FINAL_TREES)
    tree_test = (
        (1.0 - selected_weight)
        * np.quantile(squared_test, selected_quantile, axis=1)
        + selected_weight
        * np.quantile(absolute_test, selected_quantile, axis=1)
    )
    prediction = final_blend(tree_test, pair_test)
    output = submission.copy()
    output[TARGET] = prediction
    output_path = OUTPUT_DIR / "submit_v31_absolute_error_blend.csv"
    output.to_csv(output_path, index=False, encoding="utf-8")
    print("Saved:", output_path)
    print("Prediction range:", prediction.min(), prediction.max())
    display(output.head(10))



V31 Near-Full split 1/8 complete
V31 Near-Full split 2/8 complete
V31 Near-Full split 3/8 complete
V31 Near-Full split 4/8 complete
V31 Near-Full split 5/8 complete
V31 Near-Full split 6/8 complete
V31 Near-Full split 7/8 complete
V31 Near-Full split 8/8 complete


,quantile,absolute_weight,candidate_MAE,mean_improvement,worst_improvement,wins
0,0.50,0.25,0.128708,0.000092,-0.002000,5
1,0.48,0.50,0.128958,-0.000158,-0.002733,5
2,0.50,0.50,0.128850,-0.000050,-0.001867,4
3,0.48,0.25,0.128800,0.000000,-0.002600,4
4,0.48,1.00,0.128992,-0.000192,-0.002800,4
5,0.52,0.25,0.128900,-0.000100,-0.000533,3
6,0.50,0.75,0.128883,-0.000083,-0.002267,3
7,0.48,0.75,0.128925,-0.000125,-0.002533,3
8,0.54,1.00,0.129650,-0.000850,-0.004867,3
9,0.52,0.50,0.129033,-0.000233,-0.001600,2


V31 rejected: absolute-error 후보가 안정성 기준을 통과하지 못했습니다.
제출 파일을 생성하지 않습니다.
